# Lesson 09
Run lessons 1–8 first, or use 00_run_all.ipynb. This is the actual teaching source, not a placeholder.

# 第9课 衍生品定价、Greeks与离散对冲

目标：实现欧式Black–Scholes–Merton，交叉核对QuantLib、二叉树与TPU蒙特卡洛；从自融资账本解释Delta对冲频率、费用与跳跃尾损。模拟对冲是独立专题，不与股票历史策略净值混合。

d₁ = [ln(S/K) + (r − q + σ²/2)T] / (σ√T)，d₂ = d₁ − σ√T。

$C=S e^{-qT}N(d_1)-K e^{-rT}N(d_2)$；$P=K e^{-rT}N(-d_2)-S e^{-qT}N(-d_1)$。风险中性期望按r−q漂移并以r折现。Delta/Gamma/Vega/Theta分别表示价格对S、一阶Delta对S、波动率和时间的局部敏感度；Vega按波动率变化1.00、Theta按一年计，输出时说明单位。

实验：1）手算ATM无息价格、put-call parity、有限差分Greeks与QuantLib；2）TPU反向样本蒙特卡洛和CRR收敛；3）同一组GBM与跳跃路径比较1/5/21日再平衡、自融资现金利息、交易费用、末端交割；4）Cboe VIX的30日年化隐含方差，与SPX完全相同30个日历日窗、365日口径的前瞻实现方差作诊断。实现方差在未来成熟后才可用于评价，不能当当天输入。

VIX本身不可直接交易。没有历史期权bid/ask、合约、行权/到期与公司行动数据，本课不宣称完成真实期权策略回测。Delta对冲不能消除Gamma、Vega、跳跃和离散交易误差。对比VIX与SPX忽略微观结构、非交易时段分布和指数方法差异，只能作为方差溢价代理。

参考：[QuantLib](https://www.quantlib.org/docs.shtml)、[Cboe官方历史指数](https://www.cboe.com/tradable-products/vix/vix-historical-data)。


In [ ]:
# 09.01 BSM定价、Greeks、隐含波动率与QuantLib交叉核验
from scipy.stats import norm
from scipy.optimize import brentq
import QuantLib as ql

def bsm9(S,K,r,q,sigma,T,kind='call'):
    S=np.asarray(S,float);T=float(T)
    if T<=0:
        call=np.maximum(S-K,0);price=call if kind=='call' else np.maximum(K-S,0)
        delta=np.where(S>K,1.,np.where(S<K,0.,.5))-(kind=='put')
        return {'price':price,'delta':delta,'gamma':np.zeros_like(S),'vega':np.zeros_like(S),'theta':np.zeros_like(S)}
    if sigma<=0 or np.any(S<=0) or K<=0:raise ValueError('positive S,K,sigma required before expiry')
    root=np.sqrt(T);d1=(np.log(S/K)+(r-q+.5*sigma*sigma)*T)/(sigma*root);d2=d1-sigma*root;eq=np.exp(-q*T);er=np.exp(-r*T);phi=norm.pdf(d1)
    if kind=='call':
        price=S*eq*norm.cdf(d1)-K*er*norm.cdf(d2);delta=eq*norm.cdf(d1)
        theta=-S*eq*phi*sigma/(2*root)+q*S*eq*norm.cdf(d1)-r*K*er*norm.cdf(d2)
    elif kind=='put':
        price=K*er*norm.cdf(-d2)-S*eq*norm.cdf(-d1);delta=eq*(norm.cdf(d1)-1)
        theta=-S*eq*phi*sigma/(2*root)-q*S*eq*norm.cdf(-d1)+r*K*er*norm.cdf(-d2)
    else:raise ValueError('call or put')
    return {'price':price,'delta':delta,'gamma':eq*phi/(S*sigma*root),'vega':S*eq*phi*root,'theta':theta}
PARAM9={'S':100.,'K':100.,'r':.05,'q':.02,'sigma':.2,'T':1.}
BS9={k:bsm9(**PARAM9,kind=k) for k in ['call','put']};display(pd.DataFrame(BS9).T)
today9=ql.Date(2,ql.January,2020);ql.Settings.instance().evaluationDate=today9;maturity9=today9+365;dc9=ql.Actual365Fixed();calendar9=ql.NullCalendar()
spot9=ql.QuoteHandle(ql.SimpleQuote(100));rf9=ql.YieldTermStructureHandle(ql.FlatForward(today9,.05,dc9));div9=ql.YieldTermStructureHandle(ql.FlatForward(today9,.02,dc9));vol9=ql.BlackVolTermStructureHandle(ql.BlackConstantVol(today9,calendar9,.2,dc9))
process9=ql.BlackScholesMertonProcess(spot9,div9,rf9,vol9);QL9={}
for kind,opt_type in [('call',ql.Option.Call),('put',ql.Option.Put)]:
    option=ql.VanillaOption(ql.PlainVanillaPayoff(opt_type,100),ql.EuropeanExercise(maturity9));option.setPricingEngine(ql.AnalyticEuropeanEngine(process9))
    QL9[kind]={'price':option.NPV(),'delta':option.delta(),'gamma':option.gamma(),'vega':option.vega(),'theta':option.theta()}
display(pd.DataFrame(QL9).T);print('Vega per +1.00 volatility; Theta per calendar year')
FD9=[]
for kind in ['call','put']:
    p=PARAM9.copy();base=float(BS9[kind]['price']);h=.01;hs=1e-4;ht=1e-5
    price=lambda **kwargs:float(bsm9(**(p|kwargs),kind=kind)['price'])
    numerical={'delta':(price(S=100+h)-price(S=100-h))/(2*h),'gamma':(price(S=100+h)-2*base+price(S=100-h))/(h*h),
      'vega':(price(sigma=.2+hs)-price(sigma=.2-hs))/(2*hs),'theta':-(price(T=1+ht)-price(T=1-ht))/(2*ht)}
    for greek,val in numerical.items():FD9.append({'kind':kind,'Greek':greek,'analytic':float(BS9[kind][greek]),'finite_difference':val,'abs_error':abs(float(BS9[kind][greek])-val)})
FD9=pd.DataFrame(FD9);display(FD9)
IV9=brentq(lambda sig:float(bsm9(100,100,.05,.02,sig,1)['price'])-float(BS9['call']['price']),.001,3.)
print('recovered IV',IV9,'hand ATM r=q=0 call',100*(2*norm.cdf(.1)-1))
print('09.01 PASS: independent QuantLib and finite differences available')


In [ ]:
# 09.02 CRR收敛与TPU蒙特卡洛独立核验
TREE9=[]
for steps in [50,100,500,1000]:
    option=ql.VanillaOption(ql.PlainVanillaPayoff(ql.Option.Call,100),ql.EuropeanExercise(maturity9));option.setPricingEngine(ql.BinomialVanillaEngine(process9,'crr',steps))
    price=option.NPV();TREE9.append({'steps':steps,'CRR_price':price,'BSM_price':float(BS9['call']['price']),'error':price-float(BS9['call']['price'])})
@jax.jit
def monte_carlo9(key):
    z=jax.random.normal(key,(65536,),dtype=jnp.float32)
    def payoff(z):return jnp.exp(-.05)*jnp.maximum(100*jnp.exp(.05-.02-.5*.2**2+.2*z)-100,0)
    paired=(payoff(z)+payoff(-z))/2
    return paired.mean(),paired.std(ddof=1)/jnp.sqrt(paired.size)
with jax.default_device(TPU):mc_price9,mc_se9=monte_carlo9(jax.random.PRNGKey(9001))
mc_price9.block_until_ready();MC9={'estimate':float(mc_price9),'standard_error_of_independent_pairs':float(mc_se9),'antithetic_pairs':65536,'terminal_paths':131072,'device':str(mc_price9.devices()),'analytic':float(BS9['call']['price'])}
print(json.dumps(MC9,indent=2));display(pd.DataFrame(TREE9))
assert abs(MC9['estimate']-MC9['analytic'])<5*MC9['standard_error_of_independent_pairs']+.0001
print('09.02 PASS: actual TPU MC; error uses independent antithetic pairs')


In [ ]:
# 09.03 离散Delta对冲的自融资现金账本与跳跃压力
N_PATH9=512;N_STEP9=252;RATE9=.02;SIGMA9=.2;DT9=1/252;K9=100.
@jax.jit
def simulate_gbm9(key):
    z=jax.random.normal(key,(N_PATH9,N_STEP9));lr=(RATE9-.5*SIGMA9**2)*DT9+SIGMA9*jnp.sqrt(DT9)*z
    return jnp.concatenate([jnp.full((N_PATH9,1),100.),100*jnp.exp(jnp.cumsum(lr,axis=1))],axis=1)
with jax.default_device(TPU):PATH_DEVICE9=simulate_gbm9(jax.random.PRNGKey(9002))
PATH_DEVICE9.block_until_ready();PATH9=np.asarray(PATH_DEVICE9,dtype=float);JUMP9=PATH9.copy();JUMP9[:,126:]*=.8
PREMIUM9=float(bsm9(100,100,RATE9,0,SIGMA9,1)['price'])
def delta_hedge9(paths,every=1,cost_bps=0.):
    shares=np.asarray(bsm9(paths[:,0],K9,RATE9,0,SIGMA9,1)['delta']);fee=np.abs(shares)*paths[:,0]*cost_bps/10000
    cash=np.full(len(paths),PREMIUM9)-shares*paths[:,0]-fee;fees=fee.copy();record=[];residual=0.
    for i in range(1,N_STEP9+1):
        cash*=np.exp(RATE9*DT9);S=paths[:,i];before=cash+shares*S
        if i<N_STEP9 and i%every==0:
            desired=np.asarray(bsm9(S,K9,RATE9,0,SIGMA9,(N_STEP9-i)/252)['delta']);trade=desired-shares
            fee=np.abs(trade)*S*cost_bps/10000;cash-=trade*S+fee;shares=desired;fees+=fee
            residual=max(residual,float(np.max(np.abs(cash+shares*S-(before-fee)))))
        option=np.asarray(bsm9(S,K9,RATE9,0,SIGMA9,max(0,(N_STEP9-i)/252))['price'])
        record.append({'day':i,'cash_first':cash[0],'delta_first':shares[0],'spot_first':S[0],'option_first':option[0],'marked_hedge_PnL_first':cash[0]+shares[0]*S[0]-option[0]})
    fee=np.abs(shares)*paths[:,-1]*cost_bps/10000;cash+=shares*paths[:,-1]-fee;fees+=fee;shares=np.zeros(len(paths));terminal=cash-np.maximum(paths[:,-1]-K9,0)
    return {'terminal_PnL':terminal,'fees':fees,'cash_after_payoff':terminal.copy(),'shares_final':shares,'max_accounting_residual':residual,'path':pd.DataFrame(record)}
HEDGE9={};rows=[]
for scenario,paths in [('risk_neutral_GBM',PATH9),('minus20pct_jump_stress',JUMP9)]:
    for every in [1,5,21]:
        for cost in [0.,1.]:
            z=delta_hedge9(paths,every,cost);HEDGE9[(scenario,every,cost)]=z;p=z['terminal_PnL']
            rows.append({'scenario':scenario,'rebalance_every_sessions':every,'cost_bps':cost,'mean_PnL':p.mean(),'hedge_error_std':p.std(ddof=1),'q01_PnL':np.quantile(p,.01),'q05_PnL':np.quantile(p,.05),'mean_fees':z['fees'].mean(),'paths':len(p),'option_premium':PREMIUM9})
HEDGE_TABLE9=pd.DataFrame(rows);display(HEDGE_TABLE9)
fig,ax=plt.subplots(1,2,figsize=(13,4))
for every in [1,5,21]:
    for j,scenario in enumerate(['risk_neutral_GBM','minus20pct_jump_stress']):ax[j].hist(HEDGE9[(scenario,every,1.)]['terminal_PnL'],bins=35,alpha=.4,label=str(every)+' sessions')
for j,title in enumerate(['Same GBM paths: hedge error after costs','Named -20% jump: short-call hedge tail']):ax[j].legend();ax[j].set_title(title)
plt.tight_layout();plt.show();display(HEDGE9[('risk_neutral_GBM',5,1.)]['path'].head(10))
print('09.03 PASS: zero initial wealth, per-option dollar PnL, fees once, cash interest and terminal settlement')


In [ ]:
# 09.04 Cboe VIX与SPX同30日历窗方差诊断、定价对冲验收
VIX_URL9='https://cdn-api.cboe.com/api/global/us_indices/daily_prices/VIX_History.csv'
VIX_BLOB9=fetch(VIX_URL9);VIX9=pd.read_csv(io.BytesIO(VIX_BLOB9));VIX9.columns=VIX9.columns.str.strip().str.upper();VIX9['DATE']=pd.to_datetime(VIX9.DATE);VIX9=VIX9.set_index('DATE').sort_index()
spx_raw9=yf.download('^GSPC',start='2012-01-01',end='2023-01-01',auto_adjust=False,progress=False,threads=False,multi_level_index=False)
if isinstance(spx_raw9.columns,pd.MultiIndex):spx_raw9.columns=spx_raw9.columns.get_level_values(0)
SPX9=spx_raw9['Close'].dropna();SPX9.index=pd.DatetimeIndex(SPX9.index).tz_localize(None).normalize();LOGSPX9=np.log(SPX9).diff()
VAR_ROWS9=[]
for d in SPX9.loc['2014-01-01':'2022-11-30'].index:
    maturity=d+pd.Timedelta(days=30)
    if d not in VIX9.index or maturity>SPX9.index.max():continue
    future=LOGSPX9[(LOGSPX9.index>d)&(LOGSPX9.index<=maturity)].dropna()
    if len(future)<15:continue
    iv=float(VIX9.loc[d,'CLOSE'])/100
    VAR_ROWS9.append({'decision_date':d,'label_calendar_end':maturity,'last_realized_close':future.index[-1],'trading_observations':len(future),
                     'IV_annual_variance':iv*iv,'RV_annual_variance':float((future**2).sum()*365/30)})
VAR9=pd.DataFrame(VAR_ROWS9).set_index('decision_date');VAR9['variance_premium_proxy']=VAR9.IV_annual_variance-VAR9.RV_annual_variance
VIX_MANIFEST9={'url':VIX_URL9,'sha256':hashlib.sha256(VIX_BLOB9).hexdigest(),'source':'Cboe VIX official daily closes','SPX_source':'Yahoo ^GSPC index close',
 'window':'(t,t+30 calendar days], close-to-close squared log returns; 365/30 annualization','not_a_trade':'VIX is an index, no option-chain execution evidence'}
display(VAR9.head());display(VAR9.groupby(VAR9.index.year)[['IV_annual_variance','RV_annual_variance','variance_premium_proxy']].mean());print(json.dumps(VIX_MANIFEST9,indent=2))
# 重叠30日窗相关，另报非重叠近似子样本，每月首个可比日期。
NONOVERLAP9=[];next_allowed=pd.Timestamp.min
for d,r in VAR9.iterrows():
    if d>=next_allowed:NONOVERLAP9.append(d);next_allowed=r.label_calendar_end
print('non-overlap diagnostic n',len(NONOVERLAP9));display(VAR9.loc[NONOVERLAP9][['variance_premium_proxy']].describe())
fig,ax=plt.subplots(1,2,figsize=(13,4));VAR9[['IV_annual_variance','RV_annual_variance']].plot(ax=ax[0],title='Same 30-calendar-day annual variance (365 basis)')
VAR9.variance_premium_proxy.plot(ax=ax[1],title='Ex-post variance premium proxy, not strategy PnL');ax[1].axhline(0,color='k',ls='--');plt.tight_layout();plt.show()
L9_CHECKS=[]
def check9(name,ok):
    assert bool(ok),name;L9_CHECKS.append({'check':name,'status':'PASS'})
check9('put-call parity',np.isclose(float(BS9['call']['price']-BS9['put']['price']),100*np.exp(-.02)-100*np.exp(-.05),atol=1e-10))
check9('ATM hand calculation',np.isclose(float(bsm9(100,100,0,0,.2,1)['price']),100*(2*norm.cdf(.1)-1),atol=1e-10))
check9('recovered implied vol',abs(IV9-.2)<1e-9)
check9('finite-difference Greeks',FD9.abs_error.max()<.001)
for kind in ['call','put']:
    for greek in ['price','delta','gamma','vega','theta']:check9(kind+' QuantLib '+greek,abs(float(BS9[kind][greek])-QL9[kind][greek])<1e-9)
check9('expiry intrinsic',float(bsm9(120,100,.05,0,.2,0)['price'])==20)
check9('tree refinement',abs(TREE9[-1]['error'])<.01 and abs(TREE9[-1]['error'])<abs(TREE9[0]['error']))
check9('TPU Monte Carlo location',mc_price9.devices()=={TPU})
check9('Monte Carlo sampling agreement',abs(MC9['estimate']-MC9['analytic'])<5*MC9['standard_error_of_independent_pairs']+.0001)
for key,result in HEDGE9.items():
    check9(str(key)+' conserved rebalance',result['max_accounting_residual']<1e-9)
    check9(str(key)+' final zero inventory',np.allclose(result['shares_final'],0))
    check9(str(key)+' nonnegative fee',(result['fees']>=0).all())
for scenario in ['risk_neutral_GBM','minus20pct_jump_stress']:
    for every in [1,5,21]:
        check9(scenario+str(every)+' fee reduces same-path PnL',np.all(HEDGE9[(scenario,every,1.)]['terminal_PnL']<=HEDGE9[(scenario,every,0.)]['terminal_PnL']+1e-10))
check9('real VIX manifest',len(VAR9)>1000 and len(VIX_MANIFEST9['sha256'])==64)
check9('RV labels end exactly 30 calendar days',((VAR9.label_calendar_end-VAR9.index).dt.days==30).all())
check9('future maturity not treated as current feature',(VAR9.last_realized_close>VAR9.index).all() and (VAR9.last_realized_close<=VAR9.label_calendar_end).all())
L9_VALIDATION=pd.DataFrame(L9_CHECKS);display(L9_VALIDATION)
L9_CARD={'lesson':9,'status':'verified','checks':len(L9_CHECKS),'pricing':'BSM vs QuantLib, CRR and TPU MC','simulation_paths':N_PATH9,
 'observational_variance_rows':len(VAR9),'limits':['European constant-vol assumptions','hypothetical hedging paths','named deterministic jump stress','no historical option-chain backtest','VIX not tradable','overlapping ex-post variance labels']}
print('L9 FINAL PASS',json.dumps(L9_CARD,ensure_ascii=False))
